In [4]:
import opensim as osim
from osimpy import OsimGraph
from tsl_optimization import optimize_fiber_length, calc_tsl
from itertools import product
from pathlib import Path

project_dir = Path('../')
model_dir = project_dir / 'models' / 'osim'
data_dir = project_dir / 'data'
model_file = model_dir / 'rat_hindlimb_unilateral.osim'
graph = OsimGraph.from_file(str(model_file))

[info] Loaded model RatHindlimbRight from file ../models/osim/rat_hindlimb_unilateral.osim


In [5]:
import scipy.io as sio
import polars as pl
import numpy as np

control = sio.loadmat(str(data_dir / 'motion' / 'Control.mat'))
baseline_right_ik = control['Timepoints']['Baseline'][0,0]['Phases'][0,0]['RightStanceSwing'][0,0]['IK'][0,0]
avg_right_ik = baseline_right_ik['Average'][0,0]*np.pi/180
std_right_ik = baseline_right_ik['StdDev'][0,0]*np.pi/180

# Problem: Currently scipy io cannot access MATLAB string arrays. Might just have to hard code it
# ik_columns = control['Info'][0,0]['IKLabels'][0].tolist()
# print(ik_columns)
ik_columns = [
    'time', 'sacrum_pitch', 'sacrum_roll', 'sacrum_yaw', 'sacrum_x', 'sacrum_y', 'sacrum_z',
    'sacroiliac_r_flx', 'hip_r_flx', 'hip_r_add', 'hip_r_int', 'knee_r_flx', 'ankle_r_flx',
    'ankle_r_add', 'ankle_r_int', 'sacroiliac_l_flx', 'hip_l_flx', 'hip_l_add', 'hip_l_int',
    'knee_l_flx', 'ankle_l_flx', 'ankle_l_add', 'ankle_l_int'
]

avg_right_ik_df = pl.DataFrame(avg_right_ik, schema=ik_columns)
std_right_ik_df = pl.DataFrame(std_right_ik, schema=ik_columns)

coords = ["hip_r_flx", "hip_r_add", "hip_r_int", "knee_r_flx", "ankle_r_flx"]
n_coords = len(coords)
res =1
n_std = 1
n_combos_per_step = res**n_coords

avg_right_coords = avg_right_ik_df[coords]
std_right_coords = std_right_ik_df[coords]
n_rows = avg_right_coords.shape[0]
n_combos = n_rows*n_combos_per_step

ub = avg_right_coords + n_std*std_right_coords
lb = avg_right_coords - n_std*std_right_coords
dist = np.linspace(lb, ub, res) # 1st dimension is the resolution, 2nd dimension is the timesteps, 3rd dimension is the coords
coord_combos = np.array([list(product(*dist[:, i, :].T)) for i in range(n_rows)]).reshape(n_combos, n_coords)
walk_df = pl.DataFrame(coord_combos, schema=coords)

In [6]:
# Normalized fiber length range for optimization
lm_norm_range = (0.5, 1.5) 
lm_walk_range = (0.6, 1.2)

print("Getting full ROM lengths...")
results_full = graph.get_all_muscle_lengths_rom(min_points=100)
print("Got full ROM lengths")

print("Getting walk lengths...")
lengths_walk = graph.get_muscle_lengths_from_data(graph.get_muscle_names(), walk_df)
print("Got walk lengths")

tsl_results = {}
for muscle_name, lmt in results_full.items():
  muscle: osim.Muscle = graph.get_muscle(muscle_name) 
  lm_opt = float(muscle.get_optimal_fiber_length())
  lm_range = lm_opt * np.asarray(lm_norm_range)
  alpha_opt = float(muscle.get_pennation_angle_at_optimal())
  
  millard: osim.Millard2012EquilibriumMuscle = osim.Millard2012EquilibriumMuscle.safeDownCast(muscle)
  afl = millard.getActiveForceLengthCurve()
  pfl = millard.getFiberForceLengthCurve()
  tfl = millard.getTendonForceLengthCurve()

  lmt_full = np.clip(np.sort(np.unique(lmt.select(pl.col(muscle_name)))), 1.0e-6, None) # Ensure lmt is sorted, unique, and non-zero
  lm = optimize_fiber_length(lmt_full, lm_opt, alpha_opt, afl, pfl, tfl, lm_norm_range)
  tsl = calc_tsl(lmt_full, lm, lm_opt, alpha_opt, afl, pfl, tfl)

  lmt_walk = np.clip(np.sort(np.unique(lengths_walk.select(pl.col(muscle_name)))), 1.0e-6, None)
  lm_walk = optimize_fiber_length(lmt_walk, lm_opt, alpha_opt, afl, pfl, tfl, lm_walk_range)
  tsl_walk = calc_tsl(lmt_walk, lm_walk, lm_opt, alpha_opt, afl, pfl, tfl)
  

  tsl_results[muscle_name] = {
      'lmt_full': lmt_full,
      'lm_opt': lm_opt,
      'lm': lm,
      'tsl': tsl,
      'lmt_walk': lmt_walk,
      'lm_walk': lm_walk,
      'tsl_walk': tsl_walk
  }

Getting full ROM lengths...
Got full ROM lengths
Getting walk lengths...
Got walk lengths


In [7]:
#| label: tbl-tsl-comparison
from IPython.display import Markdown as md
from tabulate import tabulate
import pandas as pd
import numpy as np

johnson_params = pd.read_csv(str(data_dir / "parameters" / "johnson_2011_parameters.csv")).set_index("Abbreviation")
johnson_tsl_mm = johnson_params["lts (mm)"]
# johnson_std = johnson_params[]

tsl_rows = []
for muscle_name, res in tsl_results.items():
    abbrev = muscle_name.split("_", 1)[1] if "_" in muscle_name else muscle_name
    tsl_rows.append(
        {
            "Abbreviation": abbrev,
            "Johnson TSL (mm)": johnson_tsl_mm.get(abbrev, np.nan),
            "Full ROM TSL (mm)": np.mean(res["tsl"]) * 1000,
            "Walk TSL (mm)": np.mean(res["tsl_walk"]) * 1000,
        }
    )

tsl_df = (
    pd.DataFrame(tsl_rows)
    .sort_values("Abbreviation")
    .set_index("Abbreviation")
)
tsl_df.to_csv(data_dir / 'parameters' /'tsl_comparison.csv')

md(tabulate(tsl_df, headers="keys", tablefmt="pipe", floatfmt=".2f"))

| Abbreviation   |   Johnson TSL (mm) |   Full ROM TSL (mm) |   Walk TSL (mm) |
|:---------------|-------------------:|--------------------:|----------------:|
| AB             |               0.00 |                0.00 |            4.62 |
| AL             |               4.50 |                0.00 |            0.00 |
| AM             |               0.00 |                4.33 |            6.72 |
| BFa            |               0.00 |                0.19 |            8.14 |
| BFp            |               0.00 |                4.69 |            6.07 |
| CF             |               0.00 |                0.00 |            0.00 |
| EDL            |               9.00 |               39.69 |           41.56 |
| FDL            |               8.70 |               45.15 |           46.48 |
| FHL            |               9.70 |               40.80 |           42.28 |
| GA             |               0.00 |                3.45 |            5.04 |
| GI             |               0.00 |                8.27 |            8.23 |
| GMa            |               4.80 |                0.00 |            0.00 |
| GMe            |               0.00 |                4.71 |            7.38 |
| GMi            |               0.00 |               10.41 |           11.96 |
| GP             |               0.90 |                6.02 |            6.13 |
| GS             |               0.00 |               12.20 |           11.92 |
| IP             |               4.10 |                0.00 |            0.00 |
| LG             |               9.40 |               31.64 |           31.99 |
| MG             |               9.10 |               30.43 |           31.77 |
| OE             |               0.00 |                3.20 |            3.84 |
| OI             |               2.80 |                8.11 |            7.92 |
| Pec            |               0.00 |                7.74 |           12.09 |
| Per            |              12.70 |               36.91 |           38.11 |
| Pir            |               0.00 |                2.04 |            3.87 |
| Pla            |               6.70 |               31.70 |           31.63 |
| Pop            |               0.00 |               12.53 |           11.95 |
| QF             |               0.00 |                2.89 |            3.75 |
| RF             |               6.60 |               25.94 |           27.07 |
| SM             |               0.00 |                0.00 |            5.57 |
| STa            |               0.00 |                9.38 |           12.69 |
| STp            |               0.00 |                0.00 |            0.00 |
| Sol            |               9.50 |               21.34 |           23.39 |
| TA             |              11.90 |               22.85 |           24.01 |
| TFL            |               0.00 |                1.58 |            0.33 |
| TP             |              12.40 |               35.97 |           36.84 |
| VI             |               6.60 |               22.28 |           25.78 |
| VL             |               6.60 |               19.49 |           23.41 |
| VM             |               6.60 |               22.51 |           26.59 |